In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

from machine_intelligence.utilities.data_handling import (
    load_parquet_table,
)



In [ ]:
from machine_intelligence.utilities.viz_setup import figure_setup, VariableNames

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:

table_outputs = project_root / "table_outputs" / current_date
table_outputs.mkdir(parents=True, exist_ok=True)


Total amount of patients

In [ ]:

raw_patients_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/tables_basis/2024-06-23/patient_regular.parquet")
df_raw_patients = load_parquet_table(raw_patients_path, backend="numpy_nullable")


In [ ]:
num_raw_pat = len(df_raw_patients["patientunitstayid"])

In [ ]:
apache_pred_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/patient_unit_stay_ids/cohorts/valid_apache_prediction/2024-08-02/included_ids_valid_apache_prediction.parquet")


In [ ]:
df_apache_ids = load_parquet_table(apache_pred_path)
df_apache_ids.head()

In [ ]:
num_apache_pat = len(df_apache_ids)

In [ ]:
visit_number_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/patient_unit_stay_ids/cohorts/visit_number/2024-08-02/included_ids_visit_number.parquet")
df_visit_number_ids = load_parquet_table(visit_number_path)

In [ ]:
num_visit_pat = len(df_visit_number_ids)


In [ ]:
admit_source_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/patient_unit_stay_ids/cohorts/admit_source/2024-08-02/included_ids_admit_source.parquet")
df_admit_source_ids = load_parquet_table(admit_source_path)
num_admit_source = len(df_admit_source_ids)
num_admit_source

In [ ]:
icu_type_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/patient_unit_stay_ids/cohorts/icu_type_a/2024-08-02/included_ids_icu_type_a.parquet")
df_icu_type_ids = load_parquet_table(icu_type_path)
num_icu_type = len(df_icu_type_ids)
num_icu_type

In [ ]:
discharge_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/patient_unit_stay_ids/cohorts/discharge_offset/2024-08-02/included_ids_unit_discharge_offset.parquet")
df_discharge_ids = load_parquet_table(discharge_path)
num_discharge = len(df_discharge_ids)
num_discharge

In [ ]:
excluded_sepsis_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/patient_unit_stay_ids/cohorts/sepsis/excluded_ids_sepsis_1h_6h_168h.parquet")
df_excluded_seps_ids = load_parquet_table(excluded_sepsis_path)
num_exc_sep = len(df_excluded_seps_ids)
num_exc_sep

### Determining the reduction of patient unitstay ids

In [ ]:
df_raw_pat_ids = df_raw_patients["patientunitstayid"].drop_duplicates()
df_raw_pat_ids.head()

In [ ]:
num_patients_criteria_0 = num_raw_pat
num_patients_criteria_0

##### Valid Apache

In [ ]:
df_crit_1 = df_raw_pat_ids[df_raw_pat_ids.isin(df_apache_ids["Included ID"].to_list())]

num_patients_criteria_1 = len(df_crit_1)
num_patients_criteria_1

In [ ]:
df_crit_2 = df_crit_1[df_crit_1.isin(df_visit_number_ids["Included ID"].to_list())]

num_patients_criteria_2 = len(df_crit_2)
num_patients_criteria_2

In [ ]:
df_crit_3 = df_crit_2[df_crit_2.isin(df_admit_source_ids["Included ID"].to_list())]

num_patients_criteria_3 = len(df_crit_3)
num_patients_criteria_3

In [ ]:
df_crit_4 = df_crit_3[df_crit_3.isin(df_icu_type_ids["Included ID"].to_list())]

num_patients_criteria_4 = len(df_crit_4)
num_patients_criteria_4


In [ ]:
df_crit_5 = df_crit_4[df_crit_4.isin(df_discharge_ids["Included ID"].to_list())]

num_patients_criteria_5 = len(df_crit_5)
num_patients_criteria_5

In [ ]:
df_crit_6 = df_crit_5[~df_crit_5.isin(df_excluded_seps_ids["Excluded ID"].to_list())]

num_patients_criteria_6 = len(df_crit_6)
num_patients_criteria_6

In [ ]:
import squarify
import matplotlib.pyplot as plt
import palettable
from machine_intelligence.utilities.viz_setup import figure_setup

labels = [f"All Admissions \n{num_patients_criteria_0}", f"Crit. 1  \n{num_patients_criteria_1}",
          f"Crit. 2  \n{num_patients_criteria_2}", f"Crit. 3 \n{num_patients_criteria_3}",
          f"Crit. 4\n{num_patients_criteria_4}", f"Crit. 5 \n{num_patients_criteria_5}", f"Crit. 6\n Cohort \n{num_patients_criteria_6}"]
sizes = [num_patients_criteria_0, num_patients_criteria_1, num_patients_criteria_2, num_patients_criteria_3,
         num_patients_criteria_4, num_patients_criteria_5, num_patients_criteria_6]

# Create the treemap
# fig, ax = plt.subplots(figsize=(10, 6))
fig, ax, _, _ = figure_setup(1, 1, width_cm=10, height_cm=10, )

squarify.plot(sizes=sizes, label=labels, ax=ax, alpha=0.7, pad=True,
              color=palettable.matplotlib.Plasma_8.mpl_colors[0:])

ax.axis("off")  # Hide axes

plt.show()


if save_as_svg := True:
    fig.savefig(plot_output_dir / "cohort_patient_flow.svg", format="svg")
